# Aula 13 — Modelos de Fundação

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

**Antes de começar, ligue a GPU.** No menu do Colab: *Ambiente de
execução* → *Alterar o tipo de ambiente de execução* → **T4 GPU** →
*Salvar*. É gratuito e deixa tudo dezenas de vezes mais rápido. Sem GPU o
notebook funciona igual, só que cada resposta leva de 30 a 90 segundos.

## Parte A: Demonstração

### Baixar o modelo

A biblioteca `transformers` já vem no Colab. Na primeira vez ela baixa 1
gigabyte do Hugging Face. Sem cadastro, sem chave de API.

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NOME = "Qwen/Qwen2.5-0.5B-Instruct"
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

tokenizador = AutoTokenizer.from_pretrained(NOME)
modelo = AutoModelForCausalLM.from_pretrained(NOME).to(DISPOSITIVO)
modelo.eval()

quantos = sum(p.numel() for p in modelo.parameters())
print(f"rodando em: {DISPOSITIVO}")
print(f"{quantos:,} pesos ({quantos / 1e6:.0f} milhões)")
print(f"{quantos / 787_584:.0f} vezes o modelo que treinamos na Aula 11")
print(f"vocabulário: {len(tokenizador):,} tokens (o nosso tinha 1.024)")

### O molde de conversa

Não existe conversa. Existe um texto só, com marcadores de papel, e o
modelo continua esse texto exatamente como na Aula 12.

In [ ]:
mensagens = [
    {"role": "system", "content": "Você responde em português."},
    {"role": "user", "content": "Bom dia!"},
]

texto = tokenizador.apply_chat_template(mensagens, tokenize=False,
                                        add_generation_prompt=True)
print(texto)
print("---")
print(f"{len(tokenizador(texto).input_ids)} tokens")

### Gerar uma resposta

Os parâmetros são os mesmos da Aula 12: `temperature`, `top_p` e
`max_new_tokens`. Com `do_sample=False` ele escolhe sempre o mais
provável, e a resposta é reproduzível.

In [ ]:
def responder(mensagens, quantos=90, temperatura=0.0, semente=13):
    texto = tokenizador.apply_chat_template(mensagens, tokenize=False,
                                            add_generation_prompt=True)
    entradas = tokenizador(texto, return_tensors="pt").to(DISPOSITIVO)
    torch.manual_seed(semente)
    inicio = time.time()
    with torch.no_grad():
        if temperatura > 0:
            saida = modelo.generate(**entradas, max_new_tokens=quantos,
                                    do_sample=True, temperature=temperatura,
                                    top_p=0.9)
        else:
            saida = modelo.generate(**entradas, max_new_tokens=quantos,
                                    do_sample=False)
    novos = saida[0][entradas.input_ids.shape[1]:]
    demorou = time.time() - inicio
    print(f"[{len(novos)} tokens em {demorou:.0f}s = "
          f"{len(novos) / demorou:.1f} tokens/s]")
    return tokenizador.decode(novos, skip_special_tokens=True).strip()


print(responder([{"role": "user", "content": "Bom dia! Quem é você?"}], 60))

### O prompt é o único controle que você tem

Três peças: papel (quem ele é), contexto (o que ele precisa saber) e
tarefa com formato (o que fazer, e em que forma).

In [ ]:
pergunta = "Explique o que é uma regressão logística."

print("=== sem instrução ===")
print(responder([{"role": "user", "content": pergunta}]))

In [ ]:
papel = ("Você é professor de estatística e fala com alunos iniciantes. "
         "Use frases curtas e um exemplo concreto.")

print("=== com papel ===")
print(responder([{"role": "system", "content": papel},
                 {"role": "user", "content": pergunta}]))

Leia a resposta acima com atenção. Vocês viram regressão logística na
Aula 3, e conseguem ver os erros. Num assunto que vocês não conhecem,
a mesma frase passaria sem ninguém notar.

### Poucos exemplos valem mais que muita explicação

Quando o formato importa, mostrar funciona melhor que mandar. É o
**few-shot**: dois ou três exemplos dentro do próprio prompt.

In [ ]:
few_shot = """Classifique o comentário como POSITIVO ou NEGATIVO.

Comentário: o café estava frio.
Resposta: NEGATIVO

Comentário: atendimento rápido e simpático.
Resposta: POSITIVO

Comentário: esperei quarenta minutos e ninguém apareceu.
Resposta:"""

print(responder([{"role": "user", "content": few_shot}], 10))

### Ele não sabe dizer "não sei"

Uma pergunta de fato, com resposta conferível: Dom Casmurro tem 148
capítulos, e dá para contar em `data/machado.txt`.

In [ ]:
fato = ("Quantos capítulos tem o romance Dom Casmurro, de Machado de "
        "Assis? Responda só com o número.")

for tentativa in range(3):
    resposta = responder([{"role": "user", "content": fato}], 30,
                         temperatura=1.0, semente=tentativa)
    print(f"tentativa {tentativa + 1}: {resposta}")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

Cada geração leva de 20 a 90 segundos sem GPU. Rode uma vez e espere.

### Exercício 1: o tamanho do modelo

Rode e observe: quantos pesos, quantos tokens no vocabulário, e quantas
camadas.

In [ ]:
print(f"pesos:       {sum(p.numel() for p in modelo.parameters()):,}")
print(f"vocabulário: {len(tokenizador):,}")
print(f"camadas:     {modelo.config.num_hidden_layers}")
print(f"dimensão:    {modelo.config.hidden_size}")
print(f"cabeças:     {modelo.config.num_attention_heads}")
print(f"contexto:    {modelo.config.max_position_embeddings:,} tokens")

In [ ]:
if modelo.config.num_hidden_layers > 4:
    print(f"✅ {modelo.config.num_hidden_layers} camadas, contra 4 do nosso.")
    print(f"   Dimensão {modelo.config.hidden_size} contra 128, e contexto")
    print(f"   {modelo.config.max_position_embeddings:,} contra 128.")
else:
    print("❌ Confira se a célula que carrega o modelo rodou.")

### Exercício 2: a sua primeira pergunta

Monte `minhas_mensagens` com uma pergunta sua e gere a resposta em
`minha_resposta`. Use `responder(...)`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(minha_resposta)

In [ ]:
if isinstance(minha_resposta, str) and len(minha_resposta) > 20:
    print("✅ O modelo respondeu.")
    print("   Confira o conteúdo: você viu séries temporais na Aula 4.")
else:
    print("❌ Confira o formato: uma lista de dicionários com role e content.")

### Exercício 3: o papel muda o tom

Escreva `meu_papel`, um prompt de sistema que faça o modelo responder
como se falasse com uma criança de dez anos. Depois compare as duas
respostas para a mesma pergunta.

In [ ]:
minha_pergunta = "O que é inteligência artificial?"
sem_papel = responder([{"role": "user", "content": minha_pergunta}], 60)

In [ ]:
# SEU CODIGO AQUI

In [ ]:
com_papel = responder([{"role": "system", "content": meu_papel},
                       {"role": "user", "content": minha_pergunta}], 60)

print("=== sem papel ===")
print(sem_papel)
print()
print("=== com papel ===")
print(com_papel)

In [ ]:
if len(meu_papel) > 30 and sem_papel != com_papel:
    print("✅ As duas respostas ficaram diferentes, com os mesmos pesos.")
    print("   Todo o controle veio do texto que entrou antes da pergunta.")
else:
    print("❌ Escreva um papel mais específico: quem ele é e para quem fala.")

### Exercício 4: pedindo um formato

Peça uma resposta em formato fixo e veja se ele obedece. Complete
`meu_pedido` pedindo **exatamente três itens**, cada um começando com
um hífen.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
resposta_formatada = responder([{"role": "user", "content": meu_pedido}], 80)
print(resposta_formatada)
print()
linhas = [l for l in resposta_formatada.split("\n") if l.strip().startswith("-")]
print(f"linhas que começam com hífen: {len(linhas)}")

In [ ]:
if len(linhas) == 3:
    print("✅ Ele obedeceu desta vez.")
else:
    print(f"Ele entregou {len(linhas)} linhas em vez de 3, e isso é comum.")
    print("Um modelo de 0,5 bilhão segue conteúdo bem e formato mal.")
    print("Modelos maiores obedecem. Este não é confiável nisso.")

### Exercício 5: a temperatura

Rode e observe: a mesma instrução criativa, três vezes em cada
temperatura.

In [ ]:
criativo = "Dê um nome para uma cafeteria. Só o nome, sem explicação."

for temperatura in (0.2, 1.2):
    print(f"=== T = {temperatura} ===")
    for tentativa in range(3):
        print(" ", responder([{"role": "user", "content": criativo}], 20,
                             temperatura=temperatura, semente=100 + tentativa))
    print()

In [ ]:
print("Converse com um colega: qual das duas temperaturas você usaria para")
print("extrair a data de um contrato, e qual para pensar em nomes de produto?")

### Exercício 6: few-shot

Monte `meu_few_shot`: um prompt com **dois** exemplos resolvidos e um
terceiro em aberto. A tarefa é sua: pode ser classificar, traduzir,
extrair um dado.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(responder([{"role": "user", "content": meu_few_shot}], 15))

In [ ]:
if meu_few_shot.count("\n\n") >= 2:
    print("✅ Prompt com exemplos montado.")
    print("   Nada foi treinado: os exemplos só deixam o padrão óbvio.")
else:
    print("❌ Separe os exemplos com linha em branco, e deixe o último aberto.")

### Exercício 7: desafio, ache um erro

Faça três perguntas de fato cuja resposta você **consegue conferir**.
Guarde as perguntas em `minhas_perguntas` e veja quantas ele acerta.

Dica: perguntas com número são as melhores. Quantidade, ano, distância.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for pergunta in minhas_perguntas:
    print(f"P: {pergunta}")
    print(f"R: {responder([{'role': 'user', 'content': pergunta}], 25)}")
    print()

In [ ]:
print("As respostas certas são 26 estados mais o Distrito Federal, 1908 e 148.")
print("Confira uma a uma. Repare que ele nunca escreve 'não tenho certeza'.")

Agora, em texto: escolha uma tarefa do seu trabalho ou do seu dia a dia
em que um modelo desses ajudaria, e uma em que ele seria perigoso.
Explique a diferença entre as duas em três frases. Edite esta célula
(duplo clique nela) e escreva sua resposta no lugar deste parágrafo.